# Gold | Qualidade (valida `gold_staging` ANTES da publicação)

| Regra | Severidade |
|---|---|
| `CHAVE_NULA` / `CHAVE_DUPLICADA` nas dimensões e fatos | BLOQUEANTE |
| `TAXA_CORRESPONDENCIA_JOIN` fato×dimensão (município, UF) ≥ limiar | BLOQUEANTE |
| `SITUACAO_META_INVALIDA` (fora de `dim_situacao_meta`) | BLOQUEANTE |
| `META_NULA` / cobertura de metas | INFORMATIVA (ausência de meta é um estado, não erro) |
| `TAXA_FORA_DO_INTERVALO` (0–100) | BLOQUEANTE |

Se houver falha bloqueante, o gate levanta exceção e o notebook de promoção **não** roda — `workspace.gold` continua com a versão anterior.

In [0]:
%run ../../Monitoring/_common_monitoring

In [0]:
dbutils.widgets.text("limiar_correspondencia", "95", "Limiar mínimo de correspondência (%)")
dbutils.widgets.text("layer_dq", "GOLD", "Camada gravada em qualidade_resultados (use GOLD_DEMO no teste de falha)")
LIMIAR = float(dbutils.widgets.get("limiar_correspondencia"))
G = "workspace.gold_staging"
_m = iniciar_etapa()
dq = RegistroDQ(dbutils.widgets.get("layer_dq"), limiar_correspondencia=LIMIAR)

In [0]:
for tabela, chaves in [
    ("dim_tempo", ["ano"]), ("dim_rede", ["rede_codigo"]), ("dim_serie", ["serie_codigo"]),
    ("dim_uf", ["id_uf"]), ("dim_municipio", ["id_municipio"]),
    ("ft_resultado_municipio", ["ano", "id_municipio", "serie", "rede_codigo"]),
    ("ft_resultado_uf", ["ano", "sigla_uf", "serie", "rede_codigo"]),
    ("ft_meta_vs_resultado_municipio", ["ano", "id_municipio", "serie", "rede_codigo"]),
    ("ft_meta_vs_resultado_uf", ["ano", "sigla_uf", "serie", "rede_codigo"]),
    ("ft_meta_vs_resultado_brasil", ["ano", "rede_codigo"]),
    ("ft_taxa_alfabetizacao_ponderada", ["ano", "id_municipio", "serie", "rede_codigo"]),
]:
    dq.chave_nula_e_duplicada(f"{G}.{tabela}", chaves)

In [0]:
# Correspondência fato x dimensão
dq.taxa_correspondencia("ft_resultado_municipio x dim_municipio",
    f"{G}.ft_resultado_municipio", f"{G}.dim_municipio", "l.id_municipio = r.id_municipio")
dq.taxa_correspondencia("ft_resultado_municipio x dim_municipio (com nome/UF)",
    f"{G}.ft_resultado_municipio",
    f"(SELECT * FROM {G}.dim_municipio WHERE sigla_uf IS NOT NULL AND nome_municipio IS NOT NULL)",
    "l.id_municipio = r.id_municipio")
dq.taxa_correspondencia("ft_resultado_uf x dim_uf", f"{G}.ft_resultado_uf", f"{G}.dim_uf", "l.sigla_uf = r.sigla_uf")
dq.taxa_correspondencia("ft_resultado_municipio x dim_rede", f"{G}.ft_resultado_municipio", f"{G}.dim_rede",
    "l.rede_codigo = r.rede_codigo")

In [0]:
# Domínio de situacao_meta e intervalo das taxas
for ft in ["ft_meta_vs_resultado_municipio", "ft_meta_vs_resultado_uf", "ft_meta_vs_resultado_brasil"]:
    inval = spark.sql(f"""SELECT COUNT(*) n FROM {G}.{ft} f
        LEFT ANTI JOIN {G}.dim_situacao_meta d ON f.situacao_meta = d.situacao_meta""").first()["n"]
    dq.registrar(f"{G}.{ft}", "SITUACAO_META_INVALIDA", "BLOQUEANTE", inval, "situacao_meta fora de dim_situacao_meta")
    fora = spark.sql(f"""SELECT COUNT(*) n FROM {G}.{ft}
        WHERE taxa_alfabetizacao < 0 OR taxa_alfabetizacao > 100""").first()["n"]
    dq.registrar(f"{G}.{ft}", "TAXA_FORA_DO_INTERVALO", "BLOQUEANTE", fora, "taxa_alfabetizacao deve estar em [0,100]")
    sem_meta = spark.sql(f"""SELECT COUNT(*) n FROM {G}.{ft}
        WHERE situacao_meta = 'SEM_META'""").first()["n"]
    total = spark.table(f"{G}.{ft}").count()
    dq.registrar(f"{G}.{ft}", "META_NULA", "INFORMATIVA", sem_meta,
                 f"{sem_meta}/{total} linhas sem meta no recorte. Ausência de meta NÃO é meta zero.")

fora_pond = spark.sql(f"""SELECT COUNT(*) n FROM {G}.ft_taxa_alfabetizacao_ponderada
    WHERE taxa_alfabetizacao_ponderada < -0.000001 OR taxa_alfabetizacao_ponderada > 100.000001""").first()["n"]
dq.registrar(f"{G}.ft_taxa_alfabetizacao_ponderada", "TAXA_FORA_DO_INTERVALO", "BLOQUEANTE", fora_pond,
             "taxa ponderada deve estar em [0,100] (tolerância de 1e-6 para erro de ponto flutuante)")

# Alunos presentes sem peso: ficam fora da taxa ponderada (não podem ser ponderados). Informativa.
dbutils.widgets.text("codigo_presente", "1", "código de presenca = aluno avaliado")
sem_peso = spark.sql(f"""SELECT COUNT(*) n FROM workspace.silver.alunos
    WHERE presenca = '{dbutils.widgets.get("codigo_presente")}' AND peso_aluno IS NULL""").first()["n"]
dq.registrar("workspace.silver.alunos", "PRESENTE_SEM_PESO", "INFORMATIVA", sem_peso,
             "Alunos presentes sem peso_aluno: excluídos da taxa ponderada (não são tratados como peso zero).")

In [0]:
try:
    dq.gate()
except Exception as exc:
    registrar_execucao("GOLD", "dq_gold", _m, status="FALHA", erro=exc)
    raise
registrar_execucao("GOLD", "dq_gold", _m, status="SUCESSO")